# 评分卡建模全流程演示 (Scorecard Full Pipeline)

本notebook演示使用hscredit库完成完整的评分卡建模流程，包含两种方式：

1. **分步骤训练**：分别训练分箱器、WOE编码器、LR模型，再组装成评分卡
2. **Pipeline方式**：使用sklearn Pipeline实现全流程自动化

两种方式的最终结果完全一致，Pipeline方式代码更简洁，适合生产环境部署。

In [1]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("=" * 60)
print("hscredit 评分卡建模全流程")
print("=" * 60)

hscredit 评分卡建模全流程


## Step 1: 数据加载与探索

In [2]:
# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)

print(f"数据形状: {df.shape}")
print(f"\n列数: {len(df.columns)}")
print(f"\n目标分布:")
print(df['FPD15'].value_counts())
print(f"\n坏账率: {df['FPD15'].mean()*100:.2f}%")

数据形状: (12448, 85)

列数: 85

目标分布:
FPD15
0    11622
1      826
Name: count, dtype: int64

坏账率: 6.64%


In [3]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"特征数量: {len(feature_cols)}")

# 划分训练集和测试集
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n训练集: {X_train.shape}, 样本数: {len(y_train)}, 坏账率: {y_train.mean()*100:.2f}%")
print(f"测试集: {X_test.shape}, 样本数: {len(y_test)}, 坏账率: {y_test.mean()*100:.2f}%")

特征数量: 80

训练集: (8713, 80), 样本数: 8713, 坏账率: 6.63%
测试集: (3735, 80), 样本数: 3735, 坏账率: 6.64%


## 方式一：分步骤训练

分别训练分箱器、WOE编码器、LR模型，最后组装成评分卡。

### Step 2: 特征分箱

In [4]:
from hscredit.core.binning import OptimalBinning

# 分箱配置
n_bins = 6
method = 'best_iv'

print("开始特征分箱...")

binner = OptimalBinning(method=method, max_n_bins=n_bins)
binner.fit(X_train, y_train)

print(f"\n完成 {len(binner.bin_tables_)} 个特征的分箱")

# 查看IV排名
iv_values = []
for col in binner.bin_tables_:
    iv = binner.bin_tables_[col]['分档IV值'].sum()
    iv_values.append({'特征': col, 'IV': iv})

iv_df = pd.DataFrame(iv_values).sort_values('IV', ascending=False)
print(f"\n特征IV排名（Top 10）:")
print(iv_df.head(10).to_string(index=False))

开始特征分箱...

完成 80 个特征的分箱

特征IV排名（Top 10）:
                               特征     IV
        performance_loan_count_6m 0.4273
                         bcf_fpv1 0.4133
                        dxm_v6pro 0.3954
        performance_loan_count_3m 0.3601
                       bairong_v1 0.3493
             lender_inquiry_count 0.3456
       performance_loan_count_12m 0.3197
        performance_loan_count_1m 0.3071
                   loan_count_12m 0.3056
consumer_finance_lender_count_12m 0.3007


### Step 3: WOE编码变换

In [5]:
# 使用binner直接进行WOE转换（hscredit风格）
X_train_woe = binner.transform(X_train, metric='woe')
X_test_woe = binner.transform(X_test, metric='woe')

print(f"WOE编码后形状: {X_train_woe.shape}")
print(f"\nWOE编码结果（部分）:")
print(X_train_woe.head())

WOE编码后形状: (8713, 80)

WOE编码结果（部分）:
       bj_qy24  mobtech80  bairong_v1  xiaoniu_c3  dxm_v6pro  bcf_fpv1  apply_for_admission_confidence  apply_for_admission_score  charging_fail_count_12m  charging_fail_count_1m  ...  performance_loan_count_12m  performance_loan_count_1m  performance_loan_count_24m  performance_loan_count_3m  performance_loan_count_6m  performance_loan_sum_12m  performance_loan_sum_1m  performance_loan_sum_24m  performance_loan_sum_3m  performance_loan_sum_6m
1138   -0.6993    -0.3519     -1.0907     -1.2560    -3.5190   -0.7400                         -0.0244                    -0.4980                   0.0832                 -0.0811  ...                     -0.8258                    -0.0600                      0.5649                    -0.1893                    -0.5055                   -0.3039                  -0.4126                   -0.1724                  -0.3228                  -0.5434
12293   0.1846    -0.1576     -0.3137     -0.1988    -3.5190   -2.735

### Step 4: 特征筛选

In [6]:
from hscredit.core.selectors import IVSelector, CorrSelector

# IV筛选
iv_threshold = 0.05
iv_selector = IVSelector(threshold=iv_threshold)
X_train_iv = iv_selector.fit_transform(X_train_woe, y_train)
X_test_iv = iv_selector.transform(X_test_woe)

print(f"IV筛选（阈值>{iv_threshold}）:")
print(f"  筛选前特征数: {X_train_woe.shape[1]}")
print(f"  筛选后特征数: {X_train_iv.shape[1]}")

# 相关性筛选
corr_threshold = 0.6
corr_selector = CorrSelector(threshold=corr_threshold)
X_train_final = corr_selector.fit_transform(X_train_iv, y_train)
X_test_final = corr_selector.transform(X_test_iv)

print(f"\n相关性筛选（阈值>{corr_threshold}）:")
print(f"  筛选前特征数: {X_train_iv.shape[1]}")
print(f"  筛选后特征数: {X_train_final.shape[1]}")

# 最终特征列表
final_features = X_train_final.columns.tolist()
print(f"\n最终入模特征 ({len(final_features)}个):")
for i, f in enumerate(final_features):
    print(f"  {i+1}. {f}")

IV筛选（阈值>0.05）:
  筛选前特征数: 80
  筛选后特征数: 57

相关性筛选（阈值>0.6）:
  筛选前特征数: 57
  筛选后特征数: 22

最终入模特征 (22个):
  1. performance_loan_sum_6m
  2. lender_inquiry_count_6m
  3. lender_inquiry_count_1m
  4. hit_network_loan_lender_count
  5. credit_loan_duration
  6. consumer_finance_loan_credit_line_max
  7. consumer_finance_loan_credit_line_avg
  8. consumer_finance_loan_credit_line
  9. charging_fail_count_6m
  10. apply_for_admission_score
  11. bcf_fpv1
  12. xiaoniu_c3
  13. bairong_v1
  14. mobtech80
  15. performance_loan_sum_24m
  16. performance_loan_sum_12m
  17. normal_repayment_order_in_total_order_rate
  18. network_loan_credit_line_max
  19. loan_behavior_score
  20. loan_amt_sum_12m
  21. loan_amt_over_10k_count_12m
  22. bj_qy24


### Step 5: 模型训练

In [7]:
from hscredit.core.models import LogisticRegression
from hscredit.core.metrics import KS, AUC, Gini

# 训练逻辑回归模型
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_final, y_train)

# 预测
y_train_proba = lr_model.predict_proba(X_train_final)[:, 1]
y_test_proba = lr_model.predict_proba(X_test_final)[:, 1]

# 评估指标
train_ks = KS(y_train, y_train_proba)
train_auc = AUC(y_train, y_train_proba)
train_gini = Gini(y_train, y_train_proba)

test_ks = KS(y_test, y_test_proba)
test_auc = AUC(y_test, y_test_proba)
test_gini = Gini(y_test, y_test_proba)

print("=" * 60)
print("模型性能评估")
print("=" * 60)
print(f"{'指标':<10} {'训练集':<15} {'测试集':<15}")
print("-" * 40)
print(f"{'KS':<10} {train_ks:<15.4f} {test_ks:<15.4f}")
print(f"{'AUC':<10} {train_auc:<15.4f} {test_auc:<15.4f}")
print(f"{'Gini':<10} {train_gini:<15.4f} {test_gini:<15.4f}")

模型性能评估
指标         训练集             测试集            
----------------------------------------
KS         0.3676          0.1991         
AUC        0.7417          0.6387         
Gini       0.4835          0.2774         


### Step 6: 评分卡生成

In [8]:
from hscredit.core.models import ScoreCard

# 创建评分卡
scorecard = ScoreCard(
    binner=binner,
    lr_model=lr_model,
    pdo=50,
    rate=2,
    base_score=600,
    base_odds=(1-y_train.mean()) / y_train.mean(),
    lower=300,
    upper=800
)

# 生成评分
train_scores = scorecard.predict(X_train_final, input_type='woe')
test_scores = scorecard.predict(X_test_final, input_type='woe')

print("=" * 60)
print("评分卡参数")
print("=" * 60)
print(f"  PDO: {scorecard.pdo}")
print(f"  基础分: {scorecard.base_score}")
print(f"  分数范围: [{scorecard.lower}, {scorecard.upper}]")

print("\n评分统计:")
print(f"{'数据集':<10} {'最小值':<10} {'最大值':<10} {'均值':<10} {'标准差':<10}")
print("-" * 50)
print(f"{'训练集':<10} {train_scores.min():<10.2f} {train_scores.max():<10.2f} {train_scores.mean():<10.2f} {train_scores.std():<10.2f}")
print(f"{'测试集':<10} {test_scores.min():<10.2f} {test_scores.max():<10.2f} {test_scores.mean():<10.2f} {test_scores.std():<10.2f}")

评分卡参数
  PDO: 50
  基础分: 600
  分数范围: [300, 800]

评分统计:
数据集        最小值        最大值        均值         标准差       
--------------------------------------------------
训练集        772.07     800.00     800.00     0.30      
测试集        800.00     800.00     800.00     0.00      


### Step 7: 评分卡规则表

In [9]:
# 输出评分卡规则
scorecard_rules = scorecard.scorecard_points()

print("评分卡规则表（前20行）:")
print(scorecard_rules.head(20).to_string())

print(f"\n总行数: {len(scorecard_rules)}")

评分卡规则表（前20行）:
                             变量名称 变量含义                变量分箱     对应分数     WOE值
0         performance_loan_sum_6m              (-inf, 1.5]  -0.4500 -22.6841
1         performance_loan_sum_6m               (1.5, 3.5]   0.0100   0.5807
2         performance_loan_sum_6m               (3.5, 5.5]   0.0000   0.1305
3         performance_loan_sum_6m               (5.5, 8.5]   0.0100   0.3305
4         performance_loan_sum_6m               (8.5, 9.5]   0.0000   0.1434
5         performance_loan_sum_6m              (9.5, +inf]  -0.0100  -0.5434
6         lender_inquiry_count_6m           (-inf, 119.98]   1.7500   0.2662
7         lender_inquiry_count_6m       (119.98, 146.7099]  -0.1700  -0.0257
8         lender_inquiry_count_6m       (146.7099, 203.14]  -2.2200  -0.3382
9         lender_inquiry_count_6m         (203.14, 220.96]  -9.3700  -1.4264
10        lender_inquiry_count_6m       (220.96, 229.8699]  -2.5800  -0.3922
11        lender_inquiry_count_6m         (229.8699, +inf] -12

---

## 方式二：Pipeline全流程（推荐用于生产）

hscredit 中的分箱器、编码器、筛选器都继承自 sklearn 的 `BaseEstimator` 和 `TransformerMixin`，支持 `fit/transform/fit_transform` 接口，可以直接配合 sklearn Pipeline 使用，实现端到端自动化。

### Step 1: 简单 Pipeline（分箱 + WOE + LR）

`OptimalBinning` 的 `transform` 方法支持 `metric='woe'` 参数，直接输出 WOE 编码数据，可以直接接入 Pipeline。

In [10]:
from sklearn.pipeline import Pipeline
from hscredit.core.binning import OptimalBinning
from hscredit.core.models import LogisticRegression

# 创建Pipeline：分箱 + LR
# OptimalBinning的transform默认就是WOE转换
pipeline_simple = Pipeline([
    ('binning', OptimalBinning(method='best_iv', max_n_bins=6)),
    ('lr', LogisticRegression(max_iter=1000))
])

# 训练Pipeline
print("开始训练简单Pipeline...")
pipeline_simple.fit(X_train, y_train)

# 预测概率
y_train_proba_simple = pipeline_simple.predict_proba(X_train)[:, 1]
y_test_proba_simple = pipeline_simple.predict_proba(X_test)[:, 1]

print("\n简单Pipeline模型性能:")
print(f"  训练集 KS: {KS(y_train, y_train_proba_simple):.4f}")
print(f"  测试集 KS: {KS(y_test, y_test_proba_simple):.4f}")
print(f"  分箱后特征数: {len(pipeline_simple.named_steps['binning'].bin_tables_)}")

开始训练简单Pipeline...

简单Pipeline模型性能:
  训练集 KS: 0.3795
  测试集 KS: 0.2635
  分箱后特征数: 80


### Step 2: 完整 Pipeline（分箱 + WOE + IV筛选 + 相关性筛选 + LR）

直接使用 hscredit 的 `IVSelector` 和 `CorrSelector`，它们也都是基于 `TransformerMixin` 的。

In [11]:
from hscredit.core.selectors import IVSelector, CorrSelector
from sklearn.pipeline import Pipeline

# 方式2a：分箱 + IV筛选 + LR（不做相关性筛选）
pipeline_with_iv = Pipeline([
    ('binning', OptimalBinning(method='best_iv', max_n_bins=6)),
    ('iv_select', IVSelector(threshold=0.05)),
    ('corr_select', CorrSelector(threshold=0.6)),
    ('lr', LogisticRegression(max_iter=1000))
])

pipeline_with_iv.fit(X_train, y_train)

y_test_proba_iv = pipeline_with_iv.predict_proba(X_test)[:, 1]
print(f"Pipeline(含IV筛选)测试集 KS: {KS(y_test, y_test_proba_iv):.4f}")
print(f"  IV筛选后特征数: {len(pipeline_with_iv.named_steps['iv_select'].selected_features_)}")

Pipeline(含IV筛选)测试集 KS: 0.2295
  IV筛选后特征数: 57


### Step 3: 使用 make_pipeline 创建更简洁的 Pipeline

`make_pipeline` 会自动生成步骤名称，代码更简洁。

In [12]:
from sklearn.pipeline import make_pipeline

# 使用make_pipeline（自动生成步骤名称）
pipeline_auto = make_pipeline(
    OptimalBinning(method='best_iv', max_n_bins=6),
    IVSelector(threshold=0.05),
    CorrSelector(threshold=0.6),
    LogisticRegression(max_iter=1000)
)

pipeline_auto.fit(X_train, y_train)

y_test_proba_auto = pipeline_auto.predict_proba(X_test)[:, 1]
print(f"make_pipeline测试集 KS: {KS(y_test, y_test_proba_auto):.4f}")

# 查看Pipeline步骤
print("\nPipeline步骤:")
for name, step in pipeline_auto.named_steps.items():
    print(f"  {name}: {step.__class__.__name__}")

make_pipeline测试集 KS: 0.2295

Pipeline步骤:
  optimalbinning: OptimalBinning
  ivselector: IVSelector
  corrselector: CorrSelector
  logisticregression: LogisticRegression


### Step 4: 从 Pipeline 创建评分卡

In [17]:
# 从Pipeline中提取组件创建ScoreCard
binner_pipe = pipeline_auto.named_steps['optimalbinning']
lr_pipe = pipeline_auto.named_steps['logisticregression']

# 创建评分卡
scorecard_pipe = ScoreCard(
    pipeline=pipeline_auto,
    pdo=50,
    rate=2,
    base_score=600,
    base_odds=(1-y_train.mean()) / y_train.mean(),
    lower=300,
    upper=800
)

# 生成评分
train_scores_pipe = scorecard_pipe.predict(X_train, input_type='raw')
test_scores_pipe = scorecard_pipe.predict(X_test, input_type='raw')

print("Pipeline评分卡生成成功！")
print(f"  训练集评分范围: [{train_scores_pipe.min():.2f}, {train_scores_pipe.max():.2f}]")
print(f"  测试集评分范围: [{test_scores_pipe.min():.2f}, {test_scores_pipe.max():.2f}]")

# 输出评分卡规则
scorecard_rules_pipe = scorecard_pipe.scorecard_points()
print(f"\n评分卡规则总行数: {len(scorecard_rules_pipe)}")

Pipeline评分卡生成成功！
  训练集评分范围: [711.48, 800.00]
  测试集评分范围: [711.48, 800.00]

评分卡规则总行数: 91


In [18]:
# 对比两种方式的评分结果
print("=" * 60)
print("两种方式评分结果对比")
print("=" * 60)

# 计算相关系数
score_corr = np.corrcoef(train_scores, train_scores_pipe)[0, 1]

print(f"\n分步骤 vs Pipeline 评分相关系数: {score_corr:.6f}")
print(f"差异最大值: {np.abs(train_scores - train_scores_pipe).max():.6f}")

if score_corr > 0.999:
    print("\n✓ 两种方式生成的评分高度一致！")
else:
    print("\n✗ 评分存在差异，请检查")

两种方式评分结果对比

分步骤 vs Pipeline 评分相关系数: -0.003236
差异最大值: 88.518208

✗ 评分存在差异，请检查


### Step 5: Pipeline 超参数调优示例

In [21]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# 注意：这里只是示例，实际调参需要考虑计算成本
# 定义参数网格
param_grid = {
    'binning__max_n_bins': [4, 6, 8],
    'iv_select__threshold': [0.02, 0.05, 0.1],
    'lr__C': [0.1, 1.0, 10.0]
}

# 创建用于调参的Pipeline（使用Pipeline而不是make_pipeline，步骤名称更清晰）
pipeline_for_tuning = Pipeline([
    ('binning', OptimalBinning(method='best_iv')),
    ('iv_select', IVSelector()),
    ('lr', LogisticRegression(max_iter=1000))
])

print("Pipeline超参数调优示例")
print("参数网格:")
for k, v in param_grid.items():
    print(f"  {k}: {v}")

# 实际执行（可选，计算量大）
grid_search = GridSearchCV(
    pipeline_for_tuning, param_grid,
    scoring='roc_auc', cv=3, n_jobs=-1
)
grid_search.fit(X_train, y_train)
print(f"\n最优参数: {grid_search.best_params_}")
print(f"最优AUC: {grid_search.best_score_:.4f}")

print("\n提示：取消注释上面的代码即可执行网格搜索")

Pipeline超参数调优示例
参数网格:
  binning__max_n_bins: [4, 6, 8]
  iv_select__threshold: [0.02, 0.05, 0.1]
  lr__C: [0.1, 1.0, 10.0]

最优参数: {'binning__max_n_bins': 4, 'iv_select__threshold': 0.02, 'lr__C': 0.1}
最优AUC: 0.6926

提示：取消注释上面的代码即可执行网格搜索


---

## Step 8: 保存模型和报告

In [23]:
# 创建输出目录
output_dir = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/scorecard_report'
os.makedirs(output_dir, exist_ok=True)

# 保存模型（使用pickle）
import pickle

# 保存Pipeline方式（推荐用于生产）
with open(os.path.join(output_dir, 'scorecard_pipeline.pkl'), 'wb') as f:
    pickle.dump({
        'pipeline': scorecard_pipe,
        'scorecard': scorecard_pipe,
        'binner': binner_pipe,
        'lr_model': lr_pipe
    }, f)

# 保存分步骤方式
with open(os.path.join(output_dir, 'scorecard_stepwise.pkl'), 'wb') as f:
    pickle.dump({
        'scorecard': scorecard,
        'binner': binner,
        'lr_model': lr_model
    }, f)

# 导出评分卡规则为Excel
scorecard_rules.save(os.path.join(output_dir, '评分卡规则.xlsx'), index=False)

print("模型和规则已保存:")
print(f"  {os.path.join(output_dir, 'scorecard_pipeline.pkl')}")
print(f"  {os.path.join(output_dir, 'scorecard_stepwise.pkl')}")
print(f"  {os.path.join(output_dir, '评分卡规则.xlsx')}")

模型和规则已保存:
  /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/scorecard_report/scorecard_pipeline.pkl
  /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/scorecard_report/scorecard_stepwise.pkl
  /Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/scorecard_report/评分卡规则.xlsx


In [ ]:
# 导出分箱明细
bin_details = []
for col in final_features:
    if col in binner.bin_tables_:
        bin_table = binner.get_bin_table(col)
        bin_table = bin_table.copy()
        bin_table['特征名'] = col
        bin_details.append(bin_table)

if bin_details:
    bin_details_df = pd.concat(bin_details, ignore_index=True)
    bin_details_df.to_excel(os.path.join(output_dir, '分箱明细.xlsx'), index=False)
    print(f"分箱明细已保存: {os.path.join(output_dir, '分箱明细.xlsx')}")

---

## 总结

### 两种方式对比

| 特性 | 分步骤训练 | Pipeline方式 |
|------|-----------|-------------|
| 代码复杂度 | 较复杂，需要手动管理各组件 | 简洁，一体化管理 |
| 可解释性 | 好，可查看中间结果 | 稍差，需单独提取 |
| 生产部署 | 需要额外封装 | 直接可用 |
| 交叉验证 | 需要手动实现 | 支持sklearn CV |
| 调参优化 | 需要手动实现 | 支持GridSearch |

### 推荐使用场景

- **分步骤训练**：探索性建模、需要调试中间环节、学术研究
- **Pipeline方式**：生产环境部署、自动化流程、超参调优

### 后续优化方向

1. 使用GridSearchCV进行超参数调优
2. 添加特征重要性分析
3. 实现模型监控和预警机制
4. 构建评分卡部署API服务